In [ ]:
from imblearn.over_sampling import SMOTE
import cv2
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, mean_squared_error, mean_absolute_error
from skimage.feature import hog
from skimage.color import rgb2gray
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

def extract_hog_features(image):
    """Extract HOG features from a single image."""
    # image = skimage.transform.resize(image.numpy(), (64, 128), anti_aliasing=True)
    gray_img = rgb2gray(image)
    features = hog(
        gray_img,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )
    print('Shape: ', features.shape)

    return features

old = []
def load_images(image_dir, data):
    labels = []
    images = []
    gt_stats = []
    dirs = [image_dir + '/CS', image_dir + '/Healthy']
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png'):
                print(filename)
                # Decode filename to extract sequence number, gender, and age
                # print(filename)
                seq_number = int(filename[:7]) 
                print(seq_number)
                # Find the corresponding row in the dataset
                row = data[data['pic_id'] == seq_number]

                if row.empty:
                    print(f"No matching row found for {filename}")
                    continue

                # Drop the Disease Classification column to use all other columns as label
                label_row = row.drop(columns=['pic_id']).iloc[0]
                # 
                gt_stats.append(label_row.values)

                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))

                labels.append(idx)
                images.append(image)
    
    
    return np.array(gt_stats), np.array(labels), np.array(images)/1.0  # Normalize images
    images = [x / 1.0 for x in images]
    print(images[0])
    return gt_stats, labels, images  # Normalize images

# File paths to the dataset and image directories
# file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/X-ray Atlas/results.xlsx'
file_path = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/results3.xlsx'
data = pd.read_excel(file_path, header=0).dropna()

# train_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Train_comp+normal_half'
# val_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Val_comp+normal_half'

train_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Train_Org_Aug_Cropped2'
val_image_dir = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/Val_Org_Aug_Cropped2'

# Load training and validation images and labels
X_train, y_train, images_train = load_images(train_image_dir, data)
X_val, y_val, images_val = load_images(val_image_dir, data)
hog_train = []
hog_val = []
for x in images_train:
    hog_train.append(extract_hog_features(x))
    
for x in images_val:
    hog_val.append(extract_hog_features(x))



hog_train = np.array(hog_train)
hog_val = np.array(hog_val)

print(images_train[0])
print(X_train.shape)
print(X_val.shape)
print(images_train.shape)
print(hog_train.shape)

print(y_train.shape)
print(y_val.shape)
print(images_val.shape)
print(hog_val.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_saint_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))

    # Feature Tokenization (Dense Layer for embedding features)
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # Output Layer
    outputs = layers.Dense(1, activation='sigmoid')(x)  # Binary classification

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

def build_saint_with_cnn_model(tabular_input_dim, image_input_shape):
    # Tabular data input branch
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Dense(128, activation='relu')(tabular_input)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block for tabular data
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    y = layers.Conv2D(32, (3, 3), activation='relu')(image_input)
    y = layers.MaxPooling2D((2, 2))(y)
    y = layers.Conv2D(64, (3, 3), activation='relu')(y)
    y = layers.MaxPooling2D((2, 2))(y)
    y = layers.Flatten()(y)
    y = layers.Dense(128, activation='relu')(y)
    y = layers.Dropout(0.3)(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model
def build_saint_with_efficientnet_model(tabular_input_dim, image_input_shape):
    # Tabular data input branch
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Dense(128, activation='relu')(tabular_input)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block for tabular data
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # Image data input branch
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )   
    base_model.trainable = False  # Freeze the base model

    # Pass inputs through the base model
    y = base_model(image_input, training=False)
    y = tf.keras.layers.GlobalAveragePooling2D()(y)  # Global average pooling for 4D to 2D
    y = tf.keras.layers.Dense(512, activation='relu')(y)

    # Combine tabular and image features (Late Fusion)
    combined = layers.Concatenate()([x, y])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.3)(combined)

    # Output layer (Binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name="output")(combined)

    # Create the model
    model = models.Model(inputs=[tabular_input, image_input], outputs=outputs)
    return model
import tensorflow as tf
from tensorflow.keras import layers, models

def build_saint_with_efficientnet_hog_model(tabular_input_dim, image_input_shape, hog_input_dim):
    """Multi-modal model with Tabular (SAINT), Image (EfficientNet), and HOG feature inputs."""

    # 🔹 Tabular Data Processing (SAINT-style)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Dense(128, activation='relu')(tabular_input)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block for Tabular Data (Basic Skip Connections)
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', name="CNN_Dense2")(y)

    # 🔹 HOG Feature Processing
    hog_input = layers.Input(shape=(hog_input_dim,), name="hog_input")
    hog_x = layers.Dense(256, activation="relu", name="HOG_Dense1")(hog_input)
    hog_x = layers.Dense(128, activation="relu", name="HOG_Dense2")(hog_x)

    # 🔹 Combine Tabular, Image, and HOG Features
    combined = layers.Concatenate(name="Concatenated_Features")([x, y, hog_x])
    combined = layers.Dense(128, activation='relu', name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[tabular_input, image_input, hog_input], outputs=outputs, name="SAINT_EfficientNet_HOG_Model")

    return model
import tensorflow as tf
from tensorflow.keras import layers, models

def build_saint2_with_efficientnet_hog_model(tabular_input_dim, image_input_shape, hog_input_dim):
    """Multi-modal model with Tabular (SAINT using MultiHeadAttention), Image (EfficientNet), and HOG feature inputs."""

    # 🔹 Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten before merging with other modalities

    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', name="CNN_Dense2")(y)

    # 🔹 HOG Feature Processing
    hog_input = layers.Input(shape=(hog_input_dim,), name="hog_input")
    hog_x = layers.Dense(256, activation="relu", name="HOG_Dense1")(hog_input)
    hog_x = layers.Dense(128, activation="relu", name="HOG_Dense2")(hog_x)

    # 🔹 Combine Tabular, Image, and HOG Features
    combined = layers.Concatenate(name="Concatenated_Features")([x, y, hog_x])
    combined = layers.Dense(128, activation='relu', name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[tabular_input, image_input, hog_input], outputs=outputs, name="SAINT_EfficientNet_HOG_Model")

    return model

def build_saint3_with_efficientnet_hog_model(tabular_input_dim, image_input_shape, hog_input_dim):
    """Multi-modal model with Tabular (SAINT using MultiHeadAttention), Image (EfficientNet), and HOG feature inputs."""

    # 🔹 Tabular Data Processing (SAINT-style with Multi-Head Self-Attention)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Reshape((1, tabular_input_dim))(tabular_input)  # Convert to (batch, 1, features) for attention

    num_heads = 4
    key_dim = 32

    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        x = layers.Add()([x, residual])  # Skip connection
        x = layers.LayerNormalization()(x)  # Normalization after attention

    x = layers.Flatten()(x)  # Flatten before merging with other modalities

    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', name="CNN_Dense2")(y)

    # 🔹 HOG Feature Processing
    hog_input = layers.Input(shape=(hog_input_dim,), name="hog_input")
    hog_x = layers.Dense(256, activation="relu", name="HOG_Dense1")(hog_input)
    hog_x = layers.Dense(128, activation="relu", name="HOG_Dense2")(hog_x)

    # 🔹 Combine Tabular, Image, and HOG Features
    combined = layers.Concatenate(name="Concatenated_Features")([x, y, hog_x])
    combined = layers.Dense(128, activation='relu', name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[tabular_input, image_input, hog_input], outputs=outputs, name="SAINT_EfficientNet_HOG_Model")

    return model

def build_saint_with_vgg_hog_model(tabular_input_dim, image_input_shape, hog_input_dim):
    """Multi-modal model with Tabular (SAINT), Image (EfficientNet), and HOG feature inputs."""

    # 🔹 Tabular Data Processing (SAINT-style)
    tabular_input = layers.Input(shape=(tabular_input_dim,), name="tabular_input")
    x = layers.Dense(128, activation='relu')(tabular_input)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block for Tabular Data (Basic Skip Connections)
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.VGG19(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', name="CNN_Dense2")(y)

    # 🔹 HOG Feature Processing
    hog_input = layers.Input(shape=(hog_input_dim,), name="hog_input")
    hog_x = layers.Dense(256, activation="relu", name="HOG_Dense1")(hog_input)
    hog_x = layers.Dense(128, activation="relu", name="HOG_Dense2")(hog_x)

    # 🔹 Combine Tabular, Image, and HOG Features
    combined = layers.Concatenate(name="Concatenated_Features")([x, y, hog_x])
    combined = layers.Dense(128, activation='relu', name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[tabular_input, image_input, hog_input], outputs=outputs, name="SAINT_Vgg_HOG_Model")

    return model



In [ ]:
# input_dim = X_train.shape[1]
image_input_dim = (224,224,3)
# hog_input_dim = hog_train[0].shape[0]
# print(type(hog_input_dim))
# print(type(input_dim))
model = build_saint2_with_efficientnet_hog_model(77, image_input_dim, 26244)


X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
images_train = tf.convert_to_tensor(images_train, dtype=tf.float32)
hog_train = tf.convert_to_tensor(hog_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)

X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
images_val = tf.convert_to_tensor(images_val, dtype=tf.float32)
hog_val = tf.convert_to_tensor(hog_val, dtype=tf.float32)
y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model with class weights
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=100, restore_best_weights=True)
history = model.fit([X_train, images_train, hog_train], y_train,
                    validation_data=([X_val, images_val, hog_val], y_val),
                    epochs=20,
                    batch_size=32,
                    callbacks=[early_stop],
                    verbose=1)


model.save("saint_77_trans_vgg_gen_aug_before_crop2.keras")
model = tf.keras.models.load_model("saint_77_trans_vgg_gen_aug_before_crop2.keras")


In [ ]:
from sklearn.metrics import confusion_matrix
from matplotlib import pyplot as plt
import seaborn as sns

# model = tf.keras.models.load_model("saint_77_trans_vgg_gen_aug_before_crop2.keras")

# train_loss = history.history['loss']
# val_loss = history.history['val_loss']
# train_acc = history.history['accuracy']
# val_acc = history.history['val_accuracy']
# epochs = range(1, len(train_loss) + 1)
# # 
import json
# # # 
# # # # Save history
# with open("training_history10.json", "w") as f:
#     json.dump(history.history, f)
# 
# Load history later
with open("training_history8.json", "r") as f:
    loaded_history = json.load(f)

# Access data
train_loss = loaded_history['loss']
val_loss = loaded_history['val_loss']
train_acc = loaded_history['accuracy']
val_acc = loaded_history['val_accuracy']
epochs = range(1, len(train_loss) + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
#
# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, 'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()
# 

# model = tf.keras.models.load_model("saint_77_trans_gen_hog.keras")
# Predictions and metrics
# y_pred_train = (model.predict([X_train, images_train, hog_train]) > 0.5).astype(int)
# y_pred_test = (model.predict([X_val, images_val, hog_val]) > 0.5).astype(int)
# # y_pred_test_proba = model.predict([X_val, images_val, hog_val]).flatten()
# 
# # Compute metrics
# # print("Training Classification Report:\n", classification_report(y_train, y_pred_train, digits=4))
# class_names = ["CS", "Healthy"]
# print("Testing Classification Report:\n", classification_report(y_val, y_pred_test, digits=4))
# cm = confusion_matrix(y_val, y_pred_test)
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
# plt.xlabel('Predicted')
# plt.ylabel('True')
# plt.title('Confusion Matrix')
# plt.show()
# Save the model
